# **Data Extraction and Basic Review**

## Project
**Using Machine Learning to Predict Complaint Resolution Outcomes in Retail Banking: Evidence from Open Consumer Financial Complaint Data**

## Purpose of this notebook
This notebook prepares the first analysis-ready dataset for the capstone project. It is designed for a large CFPB complaint CSV file, so it does **not** load the full raw file into memory at once. Instead, it reads the file in chunks, filters only the records needed for the study, creates basic features, and saves a processed dataset for the next notebooks.

## Expected input file
Upload the downloaded CFPB CSV file to this Google Drive location:

`MyDrive/retail-complaint-resolution-ml/data/raw/complaints.csv`

## Main output files
- `data/processed/cfpb_retail_complaints_2017_2025.csv`
- `data/processed/cfpb_retail_complaints_sample_1000.csv`
- `data/processed/raw_complaints_sample_100.csv`
- `data/data_dictionary_simple.csv`
- `outputs/tables/raw_columns.csv`
- `outputs/tables/data_extraction_summary.csv`
- `outputs/tables/data_extraction_metadata.json`
- Count tables for product, issue, company, channel, response type, timeliness, and narrative availability

## Simple Definitions

**Raw data** means the original CFPB file before cleaning, filtering, or transformation.

**Processed data** means the cleaned and filtered dataset that will be used in later analysis.

**Retail banking complaints** means complaints related to consumer banking products such as checking accounts, savings accounts, credit cards, prepaid cards, mortgages, vehicle loans, personal loans, payday/title loans, and money transfers.

**Target variable** means the outcome the model will predict. In this project, the main target variables are `timely_response` and `company_response_to_consumer`.

**Predictor variables** means the input fields used to explain or predict the outcome, such as product, issue, company, state, submitted channel, and complaint narrative.

**Data leakage** means using information in a model that would not be available at the time of prediction. For example, `company_response_to_consumer` should not be used as a predictor when the model is trying to predict the company response outcome.

**Important memory note:** The raw CFPB CSV may be several GB. This notebook avoids `pd.read_csv(CSV_PATH)` on the full file because that can overload Colab memory.

## Step 0: Mount Google Drive

This connects Colab to your Google Drive so the notebook can read the raw CSV and save outputs into the project folder.

In [ ]:
# -------------------------------------------------------------------
# Mount Google Drive
# -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Import Libraries and Define Project Folders

This step loads the required Python libraries and creates the folder structure used by the project. The folder paths follow the GitHub structure created for the capstone.

In [ ]:
# -------------------------------------------------------------------
# Import libraries
# -------------------------------------------------------------------
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

print("Libraries loaded successfully.")

Libraries loaded successfully.


This code creates and organizes the project’s folder structure in Google Drive.

It:

* Defines the main project directory.
* Creates separate folders for raw data, processed data, output tables, and figures.
* Uses `parents=True` to create any missing parent folders.
* Uses `exist_ok=True` to avoid errors if the folders already exist.
* Defines the expected location of the raw `complaints.csv` file.
* Prints the main folder paths so they can be checked before processing.

Overall, this setup keeps the project files and outputs organized in clearly separated folders.


In [ ]:
# -------------------------------------------------------------------
# Project folders
# -------------------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/retail-complaint-resolution-ml")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

for folder in [DATA_RAW, DATA_PROCESSED, OUTPUT_TABLES, OUTPUT_FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_RAW / "complaints.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)
print("Tables output folder:", OUTPUT_TABLES)
print("Expected raw CSV path:", CSV_PATH)

Project root: /content/drive/MyDrive/retail-complaint-resolution-ml
Raw data folder: /content/drive/MyDrive/retail-complaint-resolution-ml/data/raw
Processed data folder: /content/drive/MyDrive/retail-complaint-resolution-ml/data/processed
Tables output folder: /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables
Expected raw CSV path: /content/drive/MyDrive/retail-complaint-resolution-ml/data/raw/complaints.csv


## Step 2: Validate the Raw CSV File

This step checks whether the raw CFPB CSV exists in Google Drive and records basic file information. It does not read the full file.

In [ ]:
# -------------------------------------------------------------------
# Check that raw CSV exists
# -------------------------------------------------------------------
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "The file complaints.csv was not found. Please upload it to: "
        f"{CSV_PATH}"
    )

file_size_gb = CSV_PATH.stat().st_size / (1024**3)
file_size_mb = CSV_PATH.stat().st_size / (1024**2)

print("Raw CSV found.")
print(f"File name: {CSV_PATH.name}")
print(f"File size: {file_size_gb:,.2f} GB")
print(f"Last modified timestamp: {datetime.fromtimestamp(CSV_PATH.stat().st_mtime)}")

Raw CSV found.
File name: complaints.csv
File size: 8.31 GB
Last modified timestamp: 2026-07-12 21:05:22


## Step 3: Inspect Raw Columns Without Loading the Full File

This step reads only the header row using `nrows=0`. This is safe for an 8 GB file because it only loads the column names, not the full dataset.

In [ ]:
# -------------------------------------------------------------------
# Show raw column names without loading full 8 GB file
# -------------------------------------------------------------------
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

# Read only the header row
header = pd.read_csv(CSV_PATH, nrows=0)
available_columns = list(header.columns)

columns_df = pd.DataFrame({
    "column_number": range(1, len(available_columns) + 1),
    "raw_column_name": available_columns
})

columns_path = OUTPUT_TABLES / "raw_columns.csv"
columns_df.to_csv(columns_path, index=False)

print("Raw columns saved to:", columns_path)
columns_df

Raw columns saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/raw_columns.csv


,column_number,raw_column_name
0,1,Date received
1,2,Product
2,3,Sub-product
3,4,Issue
4,5,Sub-issue
5,6,Consumer complaint narrative
6,7,Company public response
7,8,Company
8,9,State
9,10,ZIP code


## Step 4: Select Required Columns


This code verifies that the CFPB CSV file contains all the columns required for the project.

It:

* Defines the list of required complaint fields.
* Reads only the CSV header, avoiding loading the full dataset.
* Identifies the required columns available in the file.
* Creates a list of any required columns that are missing.
* Displays the total number of source columns and the selected required columns.
* Raises a `ValueError` if any required column is unavailable.
* Confirms when all required columns are present.

This validation prevents later processing errors caused by missing or changed columns in the CFPB dataset.


In [ ]:
# -------------------------------------------------------------------
# Required columns based on current CFPB file version
# -------------------------------------------------------------------

required_columns = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Company public response",
    "Company",
    "State",
    "ZIP code",
    "Tags",
    "Submitted via",
    "Date sent to company",
    "Company response to consumer",
    "Timely response?",
    "Complaint ID",
]

header = pd.read_csv(CSV_PATH, nrows=0)
available_columns = list(header.columns)

usecols = [col for col in required_columns if col in available_columns]

missing_required = [
    col for col in required_columns
    if col not in available_columns
]

print("Total columns in raw file:", len(available_columns))
print("Required columns selected:", len(usecols))
print(usecols)

if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")
else:
    print("All required columns are available.")

Total columns in raw file: 16
Required columns selected: 16
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']
All required columns are available.


## Step 5: Create a Small Raw Sample for Review

This reads only a small number of rows from the raw file. The purpose is to confirm the structure and provide a sample that can be uploaded to GitHub without exposing the full 8 GB dataset.

In [ ]:
# -------------------------------------------------------------------
# Create a small raw sample for review
# -------------------------------------------------------------------
raw_sample_path = DATA_PROCESSED / "raw_complaints_sample_100.csv"

raw_sample = pd.read_csv(
    CSV_PATH,
    usecols=usecols,
    nrows=100,
    dtype=str,
    low_memory=False
)

raw_sample.to_csv(raw_sample_path, index=False)

print("Small raw sample saved to:", raw_sample_path)
print("Raw sample shape:", raw_sample.shape)
raw_sample.head()

Small raw sample saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/data/processed/raw_complaints_sample_100.csv
Raw sample shape: (100, 16)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2024-01-25T19:56:56.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TN,38141,NaN,Web,2024-01-25T19:56:59.000Z,Closed with non-monetary relief,Yes,8223945
1,2024-01-25T19:56:56.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,TN,38141,NaN,Web,2024-01-25T19:56:59.000Z,Closed with non-monetary relief,Yes,8223946
2,2024-01-25T19:36:29.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,NaN,"EQUIFAX, INC.",TN,38141,NaN,Web,2024-01-25T19:56:42.000Z,Closed with non-monetary relief,Yes,8224101
3,2024-02-09T14:31:52.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,NaN,"EQUIFAX, INC.",TN,38141,NaN,Web,2024-02-09T14:35:59.000Z,Closed with non-monetary relief,Yes,8316429
4,2024-02-09T14:36:07.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TN,38141,NaN,Web,2024-02-09T14:36:11.000Z,Closed with non-monetary relief,Yes,8316572


## Step 6: Define the Study Scope and Filter Rules

The project focuses on retail banking-related complaints received from **January 1, 2017 through December 31, 2025**. The end date is set as an exclusive date of `2026-01-01` so all records in 2025 are included, even if they contain time values.

The product filter keeps complaints related to retail banking service delivery. Debt collection is not included in the default filter here because debt collection records can relate to many non-bank products. If debt collection is added later, it should be included only when clearly linked to a consumer banking product.

In [ ]:
# -------------------------------------------------------------------
# Study filter settings
# -------------------------------------------------------------------
START_DATE = "2017-01-01"
END_DATE_EXCLUSIVE = "2026-01-01"

start_dt = pd.Timestamp(START_DATE, tz="UTC")
end_dt = pd.Timestamp(END_DATE_EXCLUSIVE, tz="UTC")

CHUNK_SIZE = 200_000

retail_products = [
    "Checking or savings account",
    "Bank account or service",
    "Credit card",
    "Credit card or prepaid card",
    "Prepaid card",
    "Mortgage",
    "Vehicle loan or lease",
    "Consumer Loan",
    "Payday loan",
    "Payday loan, title loan, or personal loan",
    "Payday loan, title loan, personal loan, or advance loan",
    "Money transfers",
    "Money transfer, virtual currency, or money service",
    "Virtual currency",
]

print("Date range:", START_DATE, "to", END_DATE_EXCLUSIVE, "exclusive")
print("Chunk size:", f"{CHUNK_SIZE:,}")
print("Retail banking product list:")
for item in retail_products:
    print("-", item)

Date range: 2017-01-01 to 2026-01-01 exclusive
Chunk size: 200,000
Retail banking product list:
- Checking or savings account
- Bank account or service
- Credit card
- Credit card or prepaid card
- Prepaid card
- Mortgage
- Vehicle loan or lease
- Consumer Loan
- Payday loan
- Payday loan, title loan, or personal loan
- Payday loan, title loan, personal loan, or advance loan
- Money transfers
- Money transfer, virtual currency, or money service
- Virtual currency


## Step 7: Helper Functions

These functions standardize column names, safely parse CFPB timestamp fields, and create reusable count tables.

In [ ]:
# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def clean_column_names(dataframe):
    """Convert CFPB column names into Python-friendly snake_case names."""
    dataframe = dataframe.copy()
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.lower()
        .str.replace("?", "", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return dataframe


def parse_date_safe(series):
    """
    Safely parse CFPB date values.

    The CFPB file may contain timestamp values such as:
    2025-07-05 08:45:32+00:00

    utc=True handles timezone values, and format='mixed' helps when pandas sees
    more than one timestamp pattern in the file.
    """
    text_series = series.astype("string").str.strip()

    try:
        return pd.to_datetime(
            text_series,
            errors="coerce",
            utc=True,
            format="mixed"
        )
    except TypeError:
        # Fallback for older pandas versions that do not support format='mixed'
        return pd.to_datetime(
            text_series,
            errors="coerce",
            utc=True
        )


def count_table(dataframe, column, top_n=None):
    """Create a count and percentage table for one column."""
    table = (
        dataframe[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )
    table["percent"] = (table["count"] / table["count"].sum() * 100).round(2)

    if top_n is not None:
        return table.head(top_n)

    return table

## Step 8: Process the 8 GB CSV in Chunks

This is the main extraction step. The notebook reads the file in chunks, cleans each chunk, converts date fields, applies the study date range, filters retail banking products, and keeps only the filtered records.

This approach is safer than loading the full file into memory. It also records how many rows are read and how many remain after each filter.

In [ ]:
# -------------------------------------------------------------------
# Process the full CSV in chunks
# -------------------------------------------------------------------
filtered_chunks = []

total_rows_read = 0
rows_with_valid_date = 0
rows_after_date_filter = 0
rows_after_product_filter = 0
invalid_date_rows = 0

reader = pd.read_csv(
    CSV_PATH,
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
)

for i, chunk in enumerate(reader, start=1):
    total_rows_read += len(chunk)

    chunk = clean_column_names(chunk)

    # Keep raw date text for audit/debug.
    chunk["date_received_raw"] = chunk["date_received"]
    chunk["date_sent_to_company_raw"] = chunk["date_sent_to_company"]

    # Safe date conversion.
    chunk["date_received"] = parse_date_safe(chunk["date_received"])
    chunk["date_sent_to_company"] = parse_date_safe(chunk["date_sent_to_company"])

    invalid_date_rows += int(chunk["date_received"].isna().sum())

    # Remove only rows where date_received cannot be parsed.
    chunk = chunk.dropna(subset=["date_received"]).copy()
    rows_with_valid_date += len(chunk)

    # Date filter.
    chunk = chunk[
        (chunk["date_received"] >= start_dt) &
        (chunk["date_received"] < end_dt)
    ].copy()
    rows_after_date_filter += len(chunk)

    # Product filter.
    chunk = chunk[chunk["product"].isin(retail_products)].copy()
    rows_after_product_filter += len(chunk)

    if len(chunk) > 0:
        filtered_chunks.append(chunk)

    print(
        f"Chunk {i} done | "
        f"Rows read: {total_rows_read:,} | "
        f"Valid dates: {rows_with_valid_date:,} | "
        f"After date filter: {rows_after_date_filter:,} | "
        f"After product filter: {rows_after_product_filter:,}"
    )

print("Finished chunk processing.")
print("Total rows read:", f"{total_rows_read:,}")
print("Rows with invalid date_received:", f"{invalid_date_rows:,}")
print("Rows after date filter:", f"{rows_after_date_filter:,}")
print("Rows after product filter:", f"{rows_after_product_filter:,}")

Chunk 1 done | Rows read: 200,000 | Valid dates: 200,000 | After date filter: 209 | After product filter: 31
Chunk 2 done | Rows read: 400,000 | Valid dates: 400,000 | After date filter: 492 | After product filter: 78
Chunk 3 done | Rows read: 600,000 | Valid dates: 600,000 | After date filter: 618 | After product filter: 106
Chunk 4 done | Rows read: 800,000 | Valid dates: 800,000 | After date filter: 806 | After product filter: 158
Chunk 5 done | Rows read: 1,000,000 | Valid dates: 1,000,000 | After date filter: 1,089 | After product filter: 254
Chunk 6 done | Rows read: 1,200,000 | Valid dates: 1,200,000 | After date filter: 1,252 | After product filter: 313
Chunk 7 done | Rows read: 1,400,000 | Valid dates: 1,400,000 | After date filter: 83,498 | After product filter: 10,498
Chunk 8 done | Rows read: 1,600,000 | Valid dates: 1,600,000 | After date filter: 283,498 | After product filter: 34,621
Chunk 9 done | Rows read: 1,800,000 | Valid dates: 1,800,000 | After date filter: 483,498

## Step 9: Combine Filtered Chunks and Remove Duplicates

After chunk processing, the filtered pieces are combined into one dataframe. Duplicate complaint IDs are removed because `complaint_id` should uniquely identify a complaint record.

In [ ]:
# -------------------------------------------------------------------
# Combine filtered chunks and remove duplicate complaint IDs
# -------------------------------------------------------------------
if not filtered_chunks:
    raise ValueError("No records were retained after filtering. Check the product list, date range, or source file.")

df = pd.concat(filtered_chunks, ignore_index=True)

before_dedup = len(df)

if "complaint_id" in df.columns:
    df = df.drop_duplicates(subset=["complaint_id"]).copy()
else:
    df = df.drop_duplicates().copy()

after_dedup = len(df)

print("Combined filtered shape before deduplication:", f"{before_dedup:,}")
print("Final shape after deduplication:", df.shape)
print("Duplicates removed:", f"{before_dedup - after_dedup:,}")
print("Invalid date_received after combination:", df["date_received"].isna().sum())

df.head()

Combined filtered shape before deduplication: 1,289,027
Final shape after deduplication: (1289027, 18)
Duplicates removed: 0
Invalid date_received after combination: 0


,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id,date_received_raw,date_sent_to_company_raw
0,2025-07-05 08:45:32+00:00,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,Changes in loan terms during the application p...,I signed an initial disclosure for a XXXX mort...,Company believes the complaint is the result o...,"Your Favorite Lenders, Inc",OH,441XX,NaN,Web,2026-05-05 13:43:00+00:00,Closed with explanation,Yes,21636268,2025-07-05T08:45:32.000Z,2026-05-05T13:43:00.000Z
1,2025-07-11 03:39:40+00:00,"Payday loan, title loan, personal loan, or adv...",Payday loan,Charged fees or interest you didn't expect,NaN,I am filing a complaint against Integra Credit...,NaN,"Deinde Online Services, LLC",FL,33068,NaN,Web,2026-04-21 17:42:21+00:00,Closed with explanation,Yes,21447929,2025-07-11T03:39:40.000Z,2026-04-21T17:42:21.000Z
2,2025-08-11 18:59:48+00:00,Mortgage,Conventional home mortgage,Struggling to pay mortgage,Trying to communicate with the company to fix ...,NaN,NaN,"Del Toro Loan Servicing, Inc",CA,94552,Older American,Phone,2025-08-15 06:00:26+00:00,Closed with explanation,No,15198684,2025-08-11T18:59:48.000Z,2025-08-15T06:00:26.000Z
3,2025-08-17 01:15:29+00:00,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,Loan estimate or other related disclosures,"Dear all, My name is XXXX XXXX, my address is ...",Company disputes the facts presented in the co...,"RESCO, INC.",HI,96815,NaN,Web,2026-05-04 13:13:09+00:00,Closed with explanation,Yes,21848458,2025-08-17T01:15:29.000Z,2026-05-04T13:13:09.000Z
4,2025-09-25 15:49:13+00:00,Mortgage,VA mortgage,Applying for a mortgage or refinancing an exis...,Loan estimate or other related disclosures,NaN,NaN,"Mortgage Express, Inc.",GA,30662,Servicemember,Web,2026-04-30 15:44:57+00:00,Closed with explanation,Yes,21669790,2025-09-25T15:49:13.000Z,2026-04-30T15:44:57.000Z


## Step 10: Create Basic Features

These features will be used in exploratory analysis, hypothesis testing, and modeling.

- `year_received`: complaint year
- `month_received`: complaint month
- `quarter_received`: complaint quarter
- `timely_response_binary`: Yes = 1, No = 0
- `has_narrative`: published narrative available = 1, not available = 0
- `narrative_length`: length of consumer complaint narrative
- `received_to_sent_lag_days`: process lag between receiving and sending the complaint to the company

**Leakage note:** `received_to_sent_lag_days` should be treated carefully. It may be useful for process analysis, but it should not be used in an intake-stage predictive model unless it is known at the time of prediction.

In [ ]:
# -------------------------------------------------------------------
# Create basic features
# -------------------------------------------------------------------
# Remove timezone only for month/quarter period fields.
# The original date_received remains timezone-aware.
date_received_no_tz = df["date_received"].dt.tz_convert(None)

df["year_received"] = date_received_no_tz.dt.year
df["month_received"] = date_received_no_tz.dt.to_period("M").astype(str)
df["quarter_received"] = date_received_no_tz.dt.to_period("Q").astype(str)

df["timely_response_binary"] = (
    df["timely_response"]
    .astype(str)
    .str.lower()
    .str.strip()
    .map({"yes": 1, "no": 0})
)

df["has_narrative"] = df["consumer_complaint_narrative"].notna().astype(int)
df["narrative_length"] = df["consumer_complaint_narrative"].fillna("").astype(str).str.len()

df["received_to_sent_lag_days"] = (
    df["date_sent_to_company"] - df["date_received"]
).dt.days

print("Feature creation completed.")
print("Final rows:", f"{df.shape[0]:,}")
print("Final columns:", f"{df.shape[1]:,}")

df[
    [
        "date_received", "year_received", "month_received", "quarter_received",
        "product", "issue", "submitted_via", "timely_response",
        "timely_response_binary", "has_narrative", "narrative_length"
    ]
].head()

Feature creation completed.
Final rows: 1,289,027
Final columns: 25


,date_received,year_received,month_received,quarter_received,product,issue,submitted_via,timely_response,timely_response_binary,has_narrative,narrative_length
0,2025-07-05 08:45:32+00:00,2025,2025-07,2025Q3,Mortgage,Applying for a mortgage or refinancing an exis...,Web,Yes,1,1,516
1,2025-07-11 03:39:40+00:00,2025,2025-07,2025Q3,"Payday loan, title loan, personal loan, or adv...",Charged fees or interest you didn't expect,Web,Yes,1,1,998
2,2025-08-11 18:59:48+00:00,2025,2025-08,2025Q3,Mortgage,Struggling to pay mortgage,Phone,No,0,0,0
3,2025-08-17 01:15:29+00:00,2025,2025-08,2025Q3,Mortgage,Applying for a mortgage or refinancing an exis...,Web,Yes,1,1,1805
4,2025-09-25 15:49:13+00:00,2025,2025-09,2025Q3,Mortgage,Applying for a mortgage or refinancing an exis...,Web,Yes,1,0,0


## Step 11: Extraction Observations

This section produces simple observations that help explain what happened during extraction. These numbers can be used in the final report or appendix to document the data preparation process.

In [ ]:
# -------------------------------------------------------------------
# Extraction summary and observations
# -------------------------------------------------------------------
summary_items = [
    {"metric": "source_file", "value": str(CSV_PATH)},
    {"metric": "source_file_size_gb", "value": round(file_size_gb, 2)},
    {"metric": "total_rows_read", "value": int(total_rows_read)},
    {"metric": "rows_with_valid_date", "value": int(rows_with_valid_date)},
    {"metric": "rows_with_invalid_date_received", "value": int(invalid_date_rows)},
    {"metric": "rows_after_date_filter", "value": int(rows_after_date_filter)},
    {"metric": "rows_after_product_filter_before_dedup", "value": int(rows_after_product_filter)},
    {"metric": "final_rows_after_dedup", "value": int(df.shape[0])},
    {"metric": "final_columns", "value": int(df.shape[1])},
    {"metric": "records_with_narrative", "value": int(df["has_narrative"].sum())},
    {"metric": "narrative_share", "value": round(float(df["has_narrative"].mean()), 4)},
    {"metric": "minimum_date_received", "value": str(df["date_received"].min())},
    {"metric": "maximum_date_received", "value": str(df["date_received"].max())},
]

extraction_summary = pd.DataFrame(summary_items)
summary_path = OUTPUT_TABLES / "data_extraction_summary.csv"
extraction_summary.to_csv(summary_path, index=False)

print("Extraction summary saved to:", summary_path)
extraction_summary

Extraction summary saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/data_extraction_summary.csv


,metric,value
0,source_file,/content/drive/MyDrive/retail-complaint-resolu...
1,source_file_size_gb,8.31
2,total_rows_read,16545376
3,rows_with_valid_date,16545376
4,rows_with_invalid_date_received,0
5,rows_after_date_filter,11987128
6,rows_after_product_filter_before_dedup,1289027
7,final_rows_after_dedup,1289027
8,final_columns,25
9,records_with_narrative,686744


In [ ]:
# -------------------------------------------------------------------
# Quick interpretation of extraction results
# -------------------------------------------------------------------
retention_after_date = rows_after_date_filter / total_rows_read if total_rows_read else 0
retention_after_product = df.shape[0] / total_rows_read if total_rows_read else 0
narrative_share = df["has_narrative"].mean()

print("Quick observations:")
print(f"- The raw file contained {total_rows_read:,} rows read through chunk processing.")
print(f"- {rows_after_date_filter:,} rows remained after applying the study date range.")
print(f"- {df.shape[0]:,} rows remained after product filtering and deduplication.")
print(f"- The final extracted dataset represents {retention_after_product:.2%} of the raw records read.")
print(f"- Published complaint narratives are available for {narrative_share:.2%} of the final records.")
print("- Narrative availability should be treated carefully because narratives are available only when consumers consent and CFPB publishes the text.")

Quick observations:
- The raw file contained 16,545,376 rows read through chunk processing.
- 11,987,128 rows remained after applying the study date range.
- 1,289,027 rows remained after product filtering and deduplication.
- The final extracted dataset represents 7.79% of the raw records read.
- Published complaint narratives are available for 53.28% of the final records.
- Narrative availability should be treated carefully because narratives are available only when consumers consent and CFPB publishes the text.


## Step 12: Data Quality Review on the Final Extracted Dataset

This section checks missing values and key fields after filtering. The missing-value table is useful for planning later preprocessing.

In [ ]:
# -------------------------------------------------------------------
# Missing value summary for final extracted dataset
# -------------------------------------------------------------------
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
    "data_type": df.dtypes.astype(str).values
}).sort_values("missing_percent", ascending=False)

missing_path = OUTPUT_TABLES / "missing_value_summary.csv"
missing_summary.to_csv(missing_path, index=False)

print("Missing value summary saved to:", missing_path)
missing_summary.head(30)

Missing value summary saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/missing_value_summary.csv


,column,missing_count,missing_percent,data_type
10,tags,1069670,82.98,object
6,company_public_response,731646,56.76,object
5,consumer_complaint_narrative,602283,46.72,object
4,sub_issue,400657,31.08,object
8,state,28399,2.20,object
2,sub_product,7762,0.60,object
0,date_received,0,0.00,"datetime64[ns, UTC]"
3,issue,5,0.00,object
1,product,0,0.00,object
7,company,0,0.00,object


In [ ]:
# -------------------------------------------------------------------
# Key field checks
# -------------------------------------------------------------------
key_checks = pd.DataFrame([
    {"check": "final_rows", "value": df.shape[0]},
    {"check": "final_columns", "value": df.shape[1]},
    {"check": "unique_complaint_ids", "value": df["complaint_id"].nunique() if "complaint_id" in df.columns else None},
    {"check": "duplicate_complaint_ids", "value": int(df["complaint_id"].duplicated().sum()) if "complaint_id" in df.columns else None},
    {"check": "unique_products", "value": df["product"].nunique()},
    {"check": "unique_issues", "value": df["issue"].nunique()},
    {"check": "unique_companies", "value": df["company"].nunique()},
    {"check": "records_with_timely_response_missing", "value": int(df["timely_response"].isna().sum())},
    {"check": "records_with_company_response_missing", "value": int(df["company_response_to_consumer"].isna().sum())},
])

key_checks_path = OUTPUT_TABLES / "key_data_quality_checks.csv"
key_checks.to_csv(key_checks_path, index=False)

print("Key data quality checks saved to:", key_checks_path)
key_checks

Key data quality checks saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/key_data_quality_checks.csv


,check,value
0,final_rows,1289027
1,final_columns,25
2,unique_complaint_ids,1289027
3,duplicate_complaint_ids,0
4,unique_products,14
5,unique_issues,141
6,unique_companies,4617
7,records_with_timely_response_missing,0
8,records_with_company_response_missing,61


## Step 13: Create Simple Summary Count Tables

These tables will support the next exploratory analysis notebook and the final written report.

In [ ]:
# -------------------------------------------------------------------
# Create and save count tables
# -------------------------------------------------------------------
count_specs = [
    ("product", "product_counts.csv", None),
    ("issue", "issue_counts.csv", None),
    ("submitted_via", "submitted_via_counts.csv", None),
    ("timely_response", "timely_response_counts.csv", None),
    ("company_response_to_consumer", "company_response_counts.csv", None),
    ("has_narrative", "narrative_availability_counts.csv", None),
    ("company", "company_counts.csv", None),
    ("state", "state_counts.csv", None),
    ("year_received", "yearly_complaint_counts.csv", None),
]

saved_count_tables = []

for column, filename, top_n in count_specs:
    if column in df.columns:
        table = count_table(df, column, top_n=top_n)
        output_path = OUTPUT_TABLES / filename
        table.to_csv(output_path, index=False)
        saved_count_tables.append(str(output_path))

print("Saved count tables:")
for path in saved_count_tables:
    print("-", path)

Saved count tables:
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/product_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/issue_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/submitted_via_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/timely_response_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/company_response_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/narrative_availability_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/company_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/state_counts.csv
- /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/yearly_complaint_counts.csv


In [ ]:
# -------------------------------------------------------------------
# Display key count tables
# -------------------------------------------------------------------
print("Product counts")
display(count_table(df, "product"))

print("Timely response counts")
display(count_table(df, "timely_response"))

print("Company response counts")
display(count_table(df, "company_response_to_consumer", top_n=20))

print("Narrative availability")
narrative_availability = count_table(df, "has_narrative")
narrative_availability["has_narrative_label"] = narrative_availability["has_narrative"].map({
    1: "Narrative available",
    0: "Narrative not available"
})
display(narrative_availability)

Product counts


,product,count,percent
0,Checking or savings account,333982,25.91
1,Mortgage,221283,17.17
2,Credit card or prepaid card,206357,16.01
3,Credit card,194996,15.13
4,"Money transfer, virtual currency, or money ser...",164638,12.77
5,Vehicle loan or lease,85430,6.63
6,"Payday loan, title loan, or personal loan",30622,2.38
7,"Payday loan, title loan, personal loan, or adv...",24745,1.92
8,Prepaid card,15540,1.21
9,Bank account or service,6955,0.54


Timely response counts


,timely_response,count,percent
0,Yes,1270902,98.59
1,No,18125,1.41


Company response counts


,company_response_to_consumer,count,percent
0,Closed with explanation,1038396,80.56
1,Closed with monetary relief,138939,10.78
2,Closed with non-monetary relief,108765,8.44
3,Untimely response,2445,0.19
4,Closed,399,0.03
5,NaN,61,0.00
6,In progress,22,0.00


Narrative availability


,has_narrative,count,percent,has_narrative_label
0,1,686744,53.28,Narrative available
1,0,602283,46.72,Narrative not available


## Step 14: Create a Basic Data Dictionary

This data dictionary documents the most important fields used in the project. It is saved as a CSV file so it can be uploaded to GitHub and referenced in the capstone report.

In [ ]:
# -------------------------------------------------------------------
# Create full data dictionary after feature engineering
# -------------------------------------------------------------------

import pandas as pd

data_dictionary = pd.DataFrame([
    {
        "field": "date_received",
        "simple_definition": "Parsed date when the CFPB received the complaint.",
        "source_or_engineered": "Source field, parsed",
        "role_in_study": "Time control and trend variable.",
        "modeling_note": "Can be transformed into year, month, or quarter."
    },
    {
        "field": "date_received_raw",
        "simple_definition": "Original unparsed Date received value from the raw CFPB file.",
        "source_or_engineered": "Audit field",
        "role_in_study": "Used to validate date parsing.",
        "modeling_note": "Not used for modeling."
    },
    {
        "field": "product",
        "simple_definition": "Financial product selected by the consumer.",
        "source_or_engineered": "Source field",
        "role_in_study": "Main predictor for RQ1 and product-scope filtering.",
        "modeling_note": "Used as categorical predictor after filtering to retail banking products."
    },
    {
        "field": "sub_product",
        "simple_definition": "More detailed product category.",
        "source_or_engineered": "Source field",
        "role_in_study": "Optional product-level predictor.",
        "modeling_note": "Use if coverage is sufficient; combine rare categories if needed."
    },
    {
        "field": "issue",
        "simple_definition": "Main complaint issue selected by the consumer.",
        "source_or_engineered": "Source field",
        "role_in_study": "Main predictor for RQ2.",
        "modeling_note": "Used as categorical predictor; may require grouping of rare issues."
    },
    {
        "field": "sub_issue",
        "simple_definition": "More detailed complaint issue.",
        "source_or_engineered": "Source field",
        "role_in_study": "Optional detailed issue predictor.",
        "modeling_note": "Use carefully because many sub-issues may be sparse."
    },
    {
        "field": "consumer_complaint_narrative",
        "simple_definition": "Consumer-written complaint narrative published when consent was provided and CFPB removed personal information.",
        "source_or_engineered": "Source field",
        "role_in_study": "Main text field for RQ3.",
        "modeling_note": "Used for NLP features such as TF-IDF, text length, keywords, and topics."
    },
    {
        "field": "company_public_response",
        "simple_definition": "Optional public-facing company response to the complaint.",
        "source_or_engineered": "Source field",
        "role_in_study": "Post-response contextual field.",
        "modeling_note": "Do not use as intake-stage predictor because it may create data leakage."
    },
    {
        "field": "company",
        "simple_definition": "Company named in the complaint.",
        "source_or_engineered": "Source field",
        "role_in_study": "Company-level predictor/control for RQ4.",
        "modeling_note": "Use carefully; CFPB data does not include company size or market share."
    },
    {
        "field": "state",
        "simple_definition": "Consumer state associated with the complaint.",
        "source_or_engineered": "Source field",
        "role_in_study": "Geographic predictor/control for RQ4.",
        "modeling_note": "Interpret with population and market-size caveats."
    },
    {
        "field": "zip_code",
        "simple_definition": "Consumer ZIP code or partial ZIP code where available.",
        "source_or_engineered": "Source field",
        "role_in_study": "Optional geographic field.",
        "modeling_note": "Use only in aggregated form; avoid overfitting and privacy concerns."
    },
    {
        "field": "tags",
        "simple_definition": "Special CFPB tags, such as older American or servicemember, where available.",
        "source_or_engineered": "Source field",
        "role_in_study": "Optional segmentation or fairness-review variable.",
        "modeling_note": "Use carefully because tags may indicate sensitive or vulnerable groups."
    },
    {
        "field": "submitted_via",
        "simple_definition": "Channel used to submit the complaint, such as web, phone, referral, or mail.",
        "source_or_engineered": "Source field",
        "role_in_study": "Predictor/control variable.",
        "modeling_note": "Used to assess whether channel is associated with outcomes."
    },
    {
        "field": "date_sent_to_company",
        "simple_definition": "Parsed date when CFPB sent the complaint to the company.",
        "source_or_engineered": "Source field, parsed",
        "role_in_study": "Process timing variable.",
        "modeling_note": "Use carefully; may create leakage if not available at prediction time."
    },
    {
        "field": "date_sent_to_company_raw",
        "simple_definition": "Original unparsed Date sent to company value from the raw CFPB file.",
        "source_or_engineered": "Audit field",
        "role_in_study": "Used to validate date parsing.",
        "modeling_note": "Not used for modeling."
    },
    {
        "field": "company_response_to_consumer",
        "simple_definition": "Company response category provided to the consumer.",
        "source_or_engineered": "Source field",
        "role_in_study": "Primary or secondary outcome variable for response-outcome analysis.",
        "modeling_note": "Can be grouped into explanation, monetary relief, non-monetary relief, and other outcomes."
    },
    {
        "field": "timely_response",
        "simple_definition": "Indicates whether the company provided a timely response.",
        "source_or_engineered": "Source field",
        "role_in_study": "Main outcome variable for RQ1 and binary classification.",
        "modeling_note": "Original Yes/No version."
    },
    {
        "field": "complaint_id",
        "simple_definition": "Unique complaint identifier assigned by CFPB.",
        "source_or_engineered": "Source field",
        "role_in_study": "Record key.",
        "modeling_note": "Not used as a model predictor."
    },
    {
        "field": "year_received",
        "simple_definition": "Year extracted from date_received.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Time control and trend variable.",
        "modeling_note": "Can be used to control for year-level changes."
    },
    {
        "field": "month_received",
        "simple_definition": "Year-month period extracted from date_received.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Monthly trend variable.",
        "modeling_note": "Mainly used for EDA and trend analysis."
    },
    {
        "field": "quarter_received",
        "simple_definition": "Year-quarter period extracted from date_received.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Quarterly trend variable.",
        "modeling_note": "Mainly used for EDA and trend analysis."
    },
    {
        "field": "timely_response_binary",
        "simple_definition": "Numeric version of timely_response where Yes = 1 and No = 0.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Main binary target variable.",
        "modeling_note": "Used as the target for binary classification models."
    },
    {
        "field": "has_narrative",
        "simple_definition": "Indicates whether a published consumer complaint narrative is available.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Narrative availability indicator for RQ3.",
        "modeling_note": "Useful because narratives are only available when consumers consent."
    },
    {
        "field": "narrative_length",
        "simple_definition": "Character length of the consumer complaint narrative.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Simple text-based feature for RQ3.",
        "modeling_note": "Can be used with other NLP features."
    },
    {
        "field": "received_to_sent_lag_days",
        "simple_definition": "Number of days between CFPB receiving the complaint and sending it to the company.",
        "source_or_engineered": "Engineered field",
        "role_in_study": "Process timing analysis variable.",
        "modeling_note": "Use for process analysis only unless clearly available at the prediction point."
    },
])

# -------------------------------------------------------------------
# Add column availability check based on final dataframe
# -------------------------------------------------------------------

current_columns = list(df.columns)

data_dictionary["present_in_final_df"] = data_dictionary["field"].isin(current_columns)

# Optional: show any fields in df that are missing from the dictionary
missing_from_dictionary = [
    col for col in current_columns
    if col not in data_dictionary["field"].tolist()
]

print("Total fields in final dataframe:", len(current_columns))
print("Fields documented in dictionary:", len(data_dictionary))
print("Fields present in final dataframe and documented:", data_dictionary["present_in_final_df"].sum())

if missing_from_dictionary:
    print("\nFields in dataframe but missing from dictionary:")
    for col in missing_from_dictionary:
        print("-", col)
else:
    print("\nAll dataframe fields are documented in the dictionary.")

# -------------------------------------------------------------------
# Save full data dictionary
# -------------------------------------------------------------------

data_dictionary_path = PROJECT_ROOT / "data" / "data_dictionary_full.csv"
data_dictionary.to_csv(data_dictionary_path, index=False)

print("\nFull data dictionary saved to:", data_dictionary_path)

display(data_dictionary)

Total fields in final dataframe: 25
Fields documented in dictionary: 25
Fields present in final dataframe and documented: 25

All dataframe fields are documented in the dictionary.

Full data dictionary saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/data/data_dictionary_full.csv


,field,simple_definition,source_or_engineered,role_in_study,modeling_note,present_in_final_df
0,date_received,Parsed date when the CFPB received the complaint.,"Source field, parsed",Time control and trend variable.,"Can be transformed into year, month, or quarter.",True
1,date_received_raw,Original unparsed Date received value from the...,Audit field,Used to validate date parsing.,Not used for modeling.,True
2,product,Financial product selected by the consumer.,Source field,Main predictor for RQ1 and product-scope filte...,Used as categorical predictor after filtering ...,True
3,sub_product,More detailed product category.,Source field,Optional product-level predictor.,Use if coverage is sufficient; combine rare ca...,True
4,issue,Main complaint issue selected by the consumer.,Source field,Main predictor for RQ2.,Used as categorical predictor; may require gro...,True
5,sub_issue,More detailed complaint issue.,Source field,Optional detailed issue predictor.,Use carefully because many sub-issues may be s...,True
6,consumer_complaint_narrative,Consumer-written complaint narrative published...,Source field,Main text field for RQ3.,"Used for NLP features such as TF-IDF, text len...",True
7,company_public_response,Optional public-facing company response to the...,Source field,Post-response contextual field.,Do not use as intake-stage predictor because i...,True
8,company,Company named in the complaint.,Source field,Company-level predictor/control for RQ4.,Use carefully; CFPB data does not include comp...,True
9,state,Consumer state associated with the complaint.,Source field,Geographic predictor/control for RQ4.,Interpret with population and market-size cave...,True


## Step 15: Save the Processed Dataset and GitHub Sample

The full processed file may still be large, so keep it in Google Drive. Upload only the notebook, metadata, summary tables, and the 1,000-record sample to GitHub.

In [ ]:
# -------------------------------------------------------------------
# Save full processed dataset and small GitHub sample
# -------------------------------------------------------------------
processed_output_path = DATA_PROCESSED / "cfpb_retail_complaints_2017_2025.csv"
sample_1000_path = DATA_PROCESSED / "cfpb_retail_complaints_sample_1000.csv"

# Full processed dataset: keep in Google Drive.
df.to_csv(processed_output_path, index=False)

# Small sample: safe for GitHub and notebook testing.
df.sample(n=min(1000, len(df)), random_state=42).to_csv(sample_1000_path, index=False)

print("Processed dataset saved to:", processed_output_path)
print("Processed rows:", f"{df.shape[0]:,}")
print("Processed columns:", f"{df.shape[1]:,}")
print(f"Processed file size: {processed_output_path.stat().st_size / (1024**2):,.2f} MB")
print("\nSample dataset saved to:", sample_1000_path)
print(f"Sample file size: {sample_1000_path.stat().st_size / (1024**2):,.4f} MB")

Processed dataset saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/data/processed/cfpb_retail_complaints_2017_2025.csv
Processed rows: 1,289,027
Processed columns: 25
Processed file size: 1,241.13 MB

Sample dataset saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/data/processed/cfpb_retail_complaints_sample_1000.csv
Sample file size: 0.9568 MB


## Step 16: Save Reproducibility Metadata

The metadata file documents the source file, date range, filter outcomes, row counts, narrative availability, and leakage warning. This supports transparency and reproducibility.

In [ ]:
# -------------------------------------------------------------------
# Save extraction metadata
# -------------------------------------------------------------------
metadata = {
    "project": "retail-complaint-resolution-ml",
    "notebook": "01_data_extraction_fixed_colab_csv.ipynb",
    "source_file": str(CSV_PATH),
    "source_file_size_gb": round(file_size_gb, 2),
    "start_date": START_DATE,
    "end_date_exclusive": END_DATE_EXCLUSIVE,
    "chunk_size": CHUNK_SIZE,
    "total_rows_read": int(total_rows_read),
    "rows_with_valid_date": int(rows_with_valid_date),
    "rows_with_invalid_date_received": int(invalid_date_rows),
    "rows_after_date_filter": int(rows_after_date_filter),
    "rows_after_product_filter_before_dedup": int(rows_after_product_filter),
    "rows_final_after_dedup": int(df.shape[0]),
    "columns_final": int(df.shape[1]),
    "records_with_narrative": int(df["has_narrative"].sum()),
    "narrative_share": float(df["has_narrative"].mean()),
    "minimum_date_received": str(df["date_received"].min()),
    "maximum_date_received": str(df["date_received"].max()),
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "leakage_note": (
        "company_response_to_consumer should not be used as a predictor when predicting response outcome. "
        "received_to_sent_lag_days is a process variable and should not be used in intake-stage prediction unless available at the prediction point."
    ),
    "github_note": "Do not upload the full raw or processed dataset to GitHub. Upload the notebook, small sample, metadata, summary tables, and figures."
}

metadata_path = OUTPUT_TABLES / "data_extraction_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved to:", metadata_path)
metadata

Metadata saved to: /content/drive/MyDrive/retail-complaint-resolution-ml/outputs/tables/data_extraction_metadata.json


{'project': 'retail-complaint-resolution-ml',
 'notebook': '01_data_extraction_fixed_colab_csv.ipynb',
 'source_file': '/content/drive/MyDrive/retail-complaint-resolution-ml/data/raw/complaints.csv',
 'source_file_size_gb': 8.31,
 'start_date': '2017-01-01',
 'end_date_exclusive': '2026-01-01',
 'chunk_size': 200000,
 'total_rows_read': 16545376,
 'rows_with_valid_date': 16545376,
 'rows_with_invalid_date_received': 0,
 'rows_after_date_filter': 11987128,
 'rows_after_product_filter_before_dedup': 1289027,
 'rows_final_after_dedup': 1289027,
 'columns_final': 25,
 'records_with_narrative': 686744,
 'narrative_share': 0.5327615325357808,
 'minimum_date_received': '2017-01-01 00:00:00+00:00',
 'maximum_date_received': '2025-12-31 22:08:00+00:00',
 'created_at': '2026-07-20T20:56:31',
 'leakage_note': 'company_response_to_consumer should not be used as a predictor when predicting response outcome. received_to_sent_lag_days is a process variable and should not be used in intake-stage predi

## Step 17: Final File Check

This confirms that the expected output files were created. If any file shows `False`, rerun the relevant previous cell.

In [ ]:
# -------------------------------------------------------------------
# Final file check
# -------------------------------------------------------------------
files_to_check = [
    processed_output_path,
    sample_1000_path,
    raw_sample_path,
    data_dictionary_path,
    OUTPUT_TABLES / "raw_columns.csv",
    OUTPUT_TABLES / "data_extraction_summary.csv",
    OUTPUT_TABLES / "missing_value_summary.csv",
    OUTPUT_TABLES / "key_data_quality_checks.csv",
    OUTPUT_TABLES / "product_counts.csv",
    OUTPUT_TABLES / "issue_counts.csv",
    OUTPUT_TABLES / "submitted_via_counts.csv",
    OUTPUT_TABLES / "timely_response_counts.csv",
    OUTPUT_TABLES / "company_response_counts.csv",
    OUTPUT_TABLES / "narrative_availability_counts.csv",
    OUTPUT_TABLES / "company_counts.csv",
    OUTPUT_TABLES / "state_counts.csv",
    OUTPUT_TABLES / "yearly_complaint_counts.csv",
    OUTPUT_TABLES / "data_extraction_metadata.json",
]

check_results = []

for file_path in files_to_check:
    check_results.append({
        "file": str(file_path),
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / (1024**2), 4) if file_path.exists() else None
    })

check_df = pd.DataFrame(check_results)
check_df

,file,exists,size_mb
0,/content/drive/MyDrive/retail-complaint-resolu...,True,1241.1264
1,/content/drive/MyDrive/retail-complaint-resolu...,True,0.9568
2,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0400
3,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0045
4,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0003
5,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0005
6,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0008
7,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0002
8,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0005
9,/content/drive/MyDrive/retail-complaint-resolu...,True,0.0055
